# CIC-IDS2017 Wednesday: NFStream extraction and daily labeling

This notebook extracts bidirectional network flows from the original CIC-IDS2017 Wednesday PCAP capture using NFStream. It cleans the extracted records, creates the timestamp used by the original labeling procedure, applies the day-specific attack rules, assigns the remaining flows to the benign class, removes duplicate rows, and exports the daily CSV consumed by the GenIDS-CIC17 consolidation notebook.

Only the input and output paths in the configuration cell should be changed. The original PCAP capture is not distributed with this repository.

## 1. Configuration

In [ ]:
from pathlib import Path

PCAP_FILE = Path("/path/to/cic-ids2017/pcaps/03_wednesday.pcap")
OUTPUT_DIR = Path("/path/to/output/nfstream_daily_csv")
OUTPUT_FILE = OUTPUT_DIR / "03_nfstream_wednesday.csv"

IDLE_TIMEOUT = 300
ACTIVE_TIMEOUT = 20
STATISTICAL_ANALYSIS = True
DECODE_TUNNELS = True
BPF_FILTER = 'ip'
TIMEZONE = "America/Moncton"

DROP_COLUMNS = [
    "content_type",
    "user_agent",
    "server_fingerprint",
    "client_fingerprint",
    "requested_server_name",
]

## 2. Imports and helper functions

In [ ]:
import pandas as pd
import pytz
import nfstream
from nfstream import NFStreamer


def validate_input(pcap_file):
    if not pcap_file.is_file():
        raise FileNotFoundError(f"PCAP file not found: {pcap_file}")


def extract_flows(pcap_file):
    return NFStreamer(
        source=str(pcap_file),
        idle_timeout=IDLE_TIMEOUT,
        active_timeout=ACTIVE_TIMEOUT,
        statistical_analysis=STATISTICAL_ANALYSIS,
        decode_tunnels=DECODE_TUNNELS,
        bpf_filter=BPF_FILTER,
    ).to_pandas()


def prepare_flows(frame):
    prepared = frame.drop(columns=DROP_COLUMNS, errors="ignore").dropna().copy()
    if "src2dst_first_seen_ms" not in prepared.columns:
        raise ValueError("NFStream output is missing the src2dst_first_seen_ms column.")
    timestamps = pd.to_datetime(
        prepared["src2dst_first_seen_ms"], unit="ms", utc=True
    ).dt.tz_convert(pytz.timezone(TIMEZONE))
    prepared["Timestamp"] = timestamps.dt.strftime("%d/%m/%Y %I:%M")
    prepared["binary"] = ""
    prepared["multiclass"] = ""
    return prepared.reset_index(drop=True)


def class_summary(frame, column):
    return pd.DataFrame({
        "count": frame[column].value_counts(),
        "percentage": frame[column].value_counts(normalize=True).mul(100).round(2),
    })


def save_daily_flows(frame, output_file):
    output_file.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_file, index=False)
    if not output_file.is_file():
        raise OSError(f"The output file was not created: {output_file}")

## 3. Validate the input and extract NFStream flows

In [ ]:
print(f"NFStream version: {nfstream.__version__}")
validate_input(PCAP_FILE)

raw_flows = extract_flows(PCAP_FILE)
print(f"Extracted flows: {len(raw_flows):,}")
print(f"Extracted columns: {raw_flows.shape[1]}")

## 4. Clean and prepare the extracted flows

In [ ]:
daily_flows = prepare_flows(raw_flows)

print(f"Flows after cleaning: {len(daily_flows):,}")
print(f"Rows removed during cleaning: {len(raw_flows) - len(daily_flows):,}")

## 5. Apply the Wednesday labeling rules

In [ ]:
df = daily_flows

mask_slowloris = ((df["src_ip"] == '172.16.0.1') &
                  (df["dst_ip"] == '192.168.10.50') &
                  ((df['Timestamp'] == '05/07/2017 9:49') |
                   (df['Timestamp'] == '05/07/2017 10:02') |
                   (df['Timestamp'] == '05/07/2017 9:55') |
                   (df['Timestamp'] == '05/07/2017 9:48') |
                   (df['Timestamp'] == '05/07/2017 9:54') |
                   (df['Timestamp'] == '05/07/2017 10:05') |
                   (df['Timestamp'] == '05/07/2017 9:59') |
                   (df['Timestamp'] == '05/07/2017 10:08') |
                   (df['Timestamp'] == '05/07/2017 9:57') |
                   (df['Timestamp'] == '05/07/2017 9:50') |
                   (df['Timestamp'] == '05/07/2017 10:09') |
                   (df['Timestamp'] == '05/07/2017 9:51') |
                   (df['Timestamp'] == '05/07/2017 9:58') |
                   (df['Timestamp'] == '05/07/2017 10:04') |
                   (df['Timestamp'] == '05/07/2017 10:06') |
                   (df['Timestamp'] == '05/07/2017 10:01') |
                   (df['Timestamp'] == '05/07/2017 10:07') |
                   (df['Timestamp'] == '05/07/2017 9:52') |
                   (df['Timestamp'] == '05/07/2017 9:53') |
                   (df['Timestamp'] == '05/07/2017 10:00') |
                   (df['Timestamp'] == '05/07/2017 10:10') |
                   (df['Timestamp'] == '05/07/2017 10:03') |
                   (df['Timestamp'] == '05/07/2017 9:56') |
                   (df['Timestamp'] == '05/07/2017 2:24') |
                   (df['Timestamp'] == '05/07/2017 10:11') |
                   (df['Timestamp'] == '05/07/2017 2:25') |
                   (df['Timestamp'] == '05/07/2017 9:01')))


df.loc[mask_slowloris, df.multiclass.name] = 'dos_slowloris'
df.loc[mask_slowloris, df.binary.name] = 'malign'

mask_slowhttptest = ((df["src_ip"] == '172.16.0.1') &
                     (df["dst_ip"] == '192.168.10.50') &
                     ((df['Timestamp'] == '05/07/2017 10:22') |
                     (df['Timestamp'] == '05/07/2017 10:33') |
                     (df['Timestamp'] == '05/07/2017 10:15') |
                     (df['Timestamp'] == '05/07/2017 10:26') |
                     (df['Timestamp'] == '05/07/2017 10:25') |
                     (df['Timestamp'] == '05/07/2017 10:19') |
                     (df['Timestamp'] == '05/07/2017 10:32') |
                     (df['Timestamp'] == '05/07/2017 10:37') |
                     (df['Timestamp'] == '05/07/2017 10:28') |
                     (df['Timestamp'] == '05/07/2017 10:18') |
                     (df['Timestamp'] == '05/07/2017 10:24') |
                     (df['Timestamp'] == '05/07/2017 10:17') |
                     (df['Timestamp'] == '05/07/2017 10:27') |
                     (df['Timestamp'] == '05/07/2017 10:34') |
                     (df['Timestamp'] == '05/07/2017 10:16') |
                     (df['Timestamp'] == '05/07/2017 10:23') |
                     (df['Timestamp'] == '05/07/2017 10:29') |
                     (df['Timestamp'] == '05/07/2017 10:35') |
                     (df['Timestamp'] == '05/07/2017 10:36')))


df.loc[mask_slowhttptest, df.multiclass.name] = 'dos_slowhttptest'
df.loc[mask_slowhttptest, df.binary.name] = 'malign'

mask_hulk = ((df["src_ip"] == '172.16.0.1') &
             (df["dst_ip"] == '192.168.10.50') &
             ((df['Timestamp'] == '05/07/2017 10:43') |
              (df['Timestamp'] == '05/07/2017 10:44') |
              (df['Timestamp'] == '05/07/2017 10:50') |
              (df['Timestamp'] == '05/07/2017 10:57') |
              (df['Timestamp'] == '05/07/2017 10:45') |
              (df['Timestamp'] == '05/07/2017 10:48') |
              (df['Timestamp'] == '05/07/2017 10:53') |
              (df['Timestamp'] == '05/07/2017 10:55') |
              (df['Timestamp'] == '05/07/2017 10:52') |
              (df['Timestamp'] == '05/07/2017 10:47') |
              (df['Timestamp'] == '05/07/2017 10:58') |
              (df['Timestamp'] == '05/07/2017 10:59') |
              (df['Timestamp'] == '05/07/2017 10:46') |
              (df['Timestamp'] == '05/07/2017 10:49') |
              (df['Timestamp'] == '05/07/2017 10:54') |
              (df['Timestamp'] == '05/07/2017 10:56') |
              (df['Timestamp'] == '05/07/2017 10:51') |
              (df['Timestamp'] == '05/07/2017 11:00') |
              (df['Timestamp'] == '05/07/2017 11:01') |
              (df['Timestamp'] == '05/07/2017 11:06') |
              (df['Timestamp'] == '05/07/2017 11:07')))


df.loc[mask_hulk, df.multiclass.name] = 'dos_hulk'
df.loc[mask_hulk, df.binary.name] = 'malign'

mask_goldeneye = ((df["src_ip"] == '172.16.0.1') &
                  (df["dst_ip"] == '192.168.10.50') &
                  ((df['Timestamp'] == '05/07/2017 11:13') |
                   (df['Timestamp'] == '05/07/2017 11:18') |
                   (df['Timestamp'] == '05/07/2017 11:12') |
                   (df['Timestamp'] == '05/07/2017 11:17') |
                   (df['Timestamp'] == '05/07/2017 11:11') |
                   (df['Timestamp'] == '05/07/2017 11:19') |
                   (df['Timestamp'] == '05/07/2017 11:10') |
                   (df['Timestamp'] == '05/07/2017 11:14')))


df.loc[mask_goldeneye, df.multiclass.name] = 'dos_goldeneye'
df.loc[mask_goldeneye, df.binary.name] = 'malign'

mask_heartbleed = ((df["src_ip"] == '172.16.0.1') &
                   (df["dst_ip"] == '192.168.10.51') &
                   ((df['Timestamp'] == '05/07/2017 3:12') |
                    (df['Timestamp'] == '05/07/2017 3:14') |
                    (df['Timestamp'] == '05/07/2017 3:16') |
                    (df['Timestamp'] == '05/07/2017 3:18') |
                    (df['Timestamp'] == '05/07/2017 3:20') |
                    (df['Timestamp'] == '05/07/2017 3:22') |
                    (df['Timestamp'] == '05/07/2017 3:24') |
                    (df['Timestamp'] == '05/07/2017 3:26') |
                    (df['Timestamp'] == '05/07/2017 3:28') |
                    (df['Timestamp'] == '05/07/2017 3:30') |
                    (df['Timestamp'] == '05/07/2017 3:32')))


df.loc[mask_heartbleed, df.multiclass.name] = 'dos_heartbleed'
df.loc[mask_heartbleed, df.binary.name] = 'malign'

df.loc[(df['binary'] == ''), df.binary.name] = 'benign'
df.loc[(df['multiclass'] == ''), df.multiclass.name] = 'benign'

daily_flows = df

## 6. Remove duplicates and summarize the daily dataset

In [ ]:
rows_before = len(daily_flows)
daily_flows = daily_flows.drop_duplicates().reset_index(drop=True)

unlabeled = daily_flows[["binary", "multiclass"]].eq("").any(axis=1).sum()
if unlabeled:
    raise ValueError(f"Found {unlabeled} flows without complete labels.")

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Rows after duplicate removal:  {len(daily_flows):,}")
display(class_summary(daily_flows, "binary"))
display(class_summary(daily_flows, "multiclass"))

## 7. Export the daily flow file

In [ ]:
save_daily_flows(daily_flows, OUTPUT_FILE)

print(f"Daily dataset saved to: {OUTPUT_FILE.resolve()}")
print(f"Final shape: {daily_flows.shape}")
display(daily_flows.head())